# 02 — Prapelatihan (Tahap 1)

**Peran:** R2 (AI Model Engineer) · **Dataset:** Intel Berkeley Lab Data

Notebook ini menceritakan tahap prapelatihan: apa yang dicoba, bagaimana korpusnya dibangun, dan **mengapa hasilnya akhirnya tidak dipakai di produksi**.

Kode sesungguhnya berada di modul `ml/`, bukan di dalam notebook ini. Notebook berfungsi sebagai penjelasan; modul `.py` adalah sumber kebenarannya, supaya tidak ada dua versi kode yang bisa saling menyimpang.

| Berkas | Isi |
|---|---|
| `ml/preprocess/build_pretrain_windows_v2.py` | membangun korpus dari data mentah Intel Lab |
| `ml/pretrain.py` | melatih backbone GRU dengan head forecast saja |
| `ml/model.py` | definisi arsitektur |

## 1. Mengapa prapelatihan dicoba

Rencana awal proyek memakai **transfer learning dua tahap**:

1. **Prapelatihan** — backbone GRU belajar dinamika suhu umum dari data IoT publik berskala besar: inersia termal, siklus harian, karakteristik derau sensor.
2. **Fine-tuning** — bobot itu dimuat, dua head baru dipasang, lalu ditala ke domain rantai dingin memakai data sintetik.

Alasannya masuk akal: label kegagalan rantai dingin tidak tersedia secara publik, sehingga data domain terbatas. Meminjam pemahaman "bentuk kurva suhu" dari data publik yang berlimpah seharusnya membantu.

Bagian ini menguji apakah dugaan tersebut benar.

## 2. Korpus prapelatihan

Percobaan pertama memakai *Temperature Readings: IoT Devices* (India). Hasilnya mengecewakan, dan diagnosisnya mengarah ke **ukuran korpus** serta **cakupan fitur**. Korpus kemudian diganti ke **Intel Berkeley Lab Data**:

| | Korpus pertama (IoT India) | Korpus kedua (Intel Lab) |
|---|---|---|
| Jendela | 6.113 | **1.308.530** (214x) |
| Fitur terisi dari 12 | 3 | 4 (tambah kelembapan) |
| Baris mentah | 97.606 | 2.313.682 |
| Derau sensor nyata | terbatas | 18,5% outlier ekstrem |

**Keterbatasan yang tetap ada:** delapan fitur sisanya — pintu, reefer, durasi reefer, kecepatan, pengereman, radiasi matahari, suhu ambien, selisih ambien — **bernilai nol**, karena Intel Lab adalah sensor ruangan tanpa konteks kendaraan. Ini kemudian menjadi kunci penjelasan mengapa transfer learning gagal.

Pembersihan mengikuti temuan EDA R1 di `docs/dataset_card.md`: baris dengan suhu di luar 0–50 °C, kelembapan di luar 0–100%, atau `moteid` tidak valid dibuang.

In [ ]:
import numpy as np

d = np.load("../../data/processed/windows/pretrain_train.npz")
print(f"Jendela latih : {d['X'].shape[0]:,}")
print(f"Bentuk        : {d['X'].shape[1:]} (60 menit x 12 fitur)")

# Kolom yang benar-benar terisi vs yang bernilai nol
terisi = (d["X"].reshape(-1, 12).std(axis=0) > 1e-6).sum()
print(f"Fitur terisi  : {terisi} dari 12")

## 3. Menjalankan prapelatihan

```bash
python -m ml.preprocess.build_pretrain_windows_v2   # bangun korpus
python -m ml.pretrain                                # latih backbone (~80 menit CPU)
```

Konfigurasi: 200.000 jendela disampel acak dari 1,16 juta (jendela bertetangga tumpang tindih 59 dari 60 menit, sehingga informasi uniknya jauh lebih sedikit daripada jumlahnya), 10 epoch, AdamW `lr=1e-3`, loss MAE pada head forecast saja.

**Catatan tentang jumlah epoch.** Korpus ini 207 kali lebih besar daripada percobaan pertama, sehingga 10 epoch di sini setara dengan pengalaman belajar 12 kali lipat dibanding 30 epoch di korpus lama. Yang menentukan adalah *jumlah kartu × jumlah putaran*, bukan angka epoch itu sendiri.

Hasil akhir: **val MAE 0,0966** pada skala ternormalisasi, membaik 25% dibanding korpus pertama (0,1286). Backbone tersimpan sebagai `ml/reports/backbone_pretrained.pt`.

## 4. Hasil: prapelatihan tidak dipakai di produksi

Backbone di atas berhasil dilatih dengan baik, tetapi **tidak memberi manfaat** saat dipakai sebagai titik awal fine-tuning. Pengujian lengkapnya ada di `03_finetune.ipynb` (Ablation B); ringkasannya:

| | Fine-tuned | Dilatih dari nol |
|---|---|---|
| Loss latih akhir | **2,99** | 3,58 |
| Loss validasi akhir | 4,18 | **2,78** |

Varian pretrained mencapai loss latih lebih rendah tetapi loss validasi lebih tinggi — **pola overfitting yang khas**.

**Penjelasannya.** Backbone belajar pada dunia di mana 8 dari 12 fitur selalu nol. Bobot GRU-nya terspesialisasi pada distribusi masukan yang timpang itu. Saat fine-tuning, spesialisasi tersebut justru harus dilupakan lebih dulu — dan karena learning rate backbone sengaja dipelankan untuk melindungi bekalnya, proses melupakan berjalan lambat.

**Kesimpulan yang dilaporkan apa adanya:** pada domain ini, prapelatihan bukan sekadar sia-sia, melainkan merugikan. Model produksi karena itu dilatih dari nol.

Skrip dan korpus prapelatihan tetap dipertahankan di repositori sebagai bukti eksperimen, bukan sebagai bagian jalur produksi.

In [ ]:
from IPython.display import Image, display

# Bukti visual Ablation B -- dibahas lengkap di 03_finetune.ipynb
display(Image(filename="../reports/loss_curves.png"))

## 5. Apa yang akan dicoba bila waktu memungkinkan

Kegagalan di atas mengarah ke satu penyebab yang jelas: **cakupan fitur korpus prapelatihan**. Perbaikan yang paling menjanjikan bukan menambah data, melainkan menambah kolom yang terisi.

Satu gagasan yang layak diuji: Intel Lab memiliki 53 sensor di satu gedung yang sama pada waktu yang sama. Sensor dekat jendela dapat berperan sebagai `ambient_c`, sensor interior sebagai `temp_c`, sehingga `delta_ambient` ikut terisi — dan korelasinya **nyata secara fisik**, bukan hasil menempelkan sumber data yang tidak berhubungan.

Menempelkan fitur dari sumber acak (misal cuaca Jakarta dengan data mengemudi Spanyol) sengaja **tidak** dilakukan: fitur bernilai nol bersifat netral, sementara fitur tempelan yang tidak berkorelasi mengajarkan hubungan yang salah — dan itu lebih buruk daripada tidak mengajarkan apa pun.